# accDM golden-file regression test

Pins a fiducial accDM model and stores reference **background**, **derived parameters**, **CMB C_l** and **P(k)** to disk, then asserts the current build reproduces them within tolerance. This is the guard against *silent numerical regressions* (the 'why is there less accDM density now' kind) across future edits to background / perturbations / input / shooting.

**Modes** (set `MODE` in the setup cell):
- `auto` (default): generate the baseline if missing, otherwise check against it.
- `generate`: deliberately (re)write the baseline. Do this only after a *vetted* physics change, then commit the new golden files.
- `check`: always compare; error if the baseline is missing.

Hybrid output: a quantitative PASS/FAIL table (with hard asserts) plus diagnostic deviation plots.

In [13]:
import os, json, hashlib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from classy import Class

# ---- Mode & paths -------------------------------------------------
#   'auto'     -> generate the golden file if missing, else check.
#   'generate' -> (re)write the baseline (overwrites! commit afterwards).
#   'check'    -> always compare; error if the baseline is missing.
MODE = 'auto'

GOLDEN_DIR  = Path('golden')
GOLDEN_DIR.mkdir(exist_ok=True)
GOLDEN_NPZ  = GOLDEN_DIR / 'regression_accDM.npz'
GOLDEN_JSON = GOLDEN_DIR / 'regression_accDM.json'

# ---- Tolerances (relative) --------------------------------
TOL = {
    'scalar': 1e-4,   # derived params (sigma8, h, Omega_*, theta_s, ...)
    'bg':     1e-4,   # background quantities at fixed z nodes
    'cl':     1e-3,   # CMB C_l (max-normalised dev over ell>=2)
    'pk':     1e-3,   # P(k) at fixed k nodes
}

LMAX = 2500
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

## Fiducial model

Planck-2018 base cosmology plus a single accDM ncdm species (Simpson quadrature, odd bin count). A realistic `kappa` is used on purpose - the `kappa ~ 100` regime is numerically stiff and not a good regression anchor. Edit the `accDM fiducial` block to match your production configuration; the parameter fingerprint below protects you from mistaking a parameter change for a code regression.

In [ ]:
# ---- Planck 2018 base cosmology ----------------------------------
omega_b    = 0.022383
omega_cdm0 = 0.12011
A_s        = 2.1005829616811546e-9
n_s        = 0.96605
tau_reio   = 0.0543
H0         = 67.32

base_params = {
    'omega_b':   omega_b,
    'omega_cdm': omega_cdm0,
    'H0':        H0,
    'A_s':       A_s,
    'n_s':       n_s,
    'tau_reio':  tau_reio,
}

# ---- accDM fiducial (EDIT to match your production config) --------
mass  = 1e16     # GeV (parent == daughter mass in the minimal model)
f_acc = 0.1      # rho_acc / rho_cdm in the far past
kappa = 12.1     # realistic; avoid the stiff kappa~100 regime here
a_t   = 0.133     # transition scale factor
#eta   = 0.1      # energy gain: E_wdm = m_wdm (1+eta)
a_rec = 1.0 / (1.0 + 1090.0)

# keep total CDM(+acc) fixed at recombination (same convention as bg_tests_w_Q)
omega_cdm_rescaled = omega_cdm0 * (1 + f_acc*(1-a_rec**kappa)/(1+(a_rec/a_t)**kappa))**(-1)

accdm_params = {
    'output':        'tCl,pCl,lCl,mPk',
    'lensing':       'yes',
    'l_max_scalars': LMAX,
    'P_k_max_1/Mpc': 10.0,
    'z_max_pk':      0.0,
    'omega_cdm':     omega_cdm_rescaled,
    'vary_Gamma_acc': 'yes',
    'kappa_acc':      kappa,
    'a_t_acc':        a_t,
    'f_acc':          f_acc,
    #'eta_acc':        eta,
    'm_acc_in_GeV':   mass,
    'm_cdm_in_GeV':   mass,
    'N_ncdm':         2,
    'deg_ncdm':       "3, 1",
    'm_ncdm':         f"0.02, {mass}*1e9",
    'T_ncdm':         "0.71611, 1",
    'N_ur':           0.0046,
    'ncdm_quadrature_strategy': "0, 5",
    # 'ncdm_N_momentum_bins':     "15, 101",
    # 'accdm_q_bins_per_decade': 100, 
    # 'accdm_q_number_tol': 1e-8,
    'reionization_z_start_max': 80,
    'evolver': 0,   # 0 = rk, 1 = ndf15 unusable right now
    'background_Nloga': 40000,  # decrease background sampling for testing (default: 40000)
    "ncdm_fluid_approximation": 3,
}

full_params = {**base_params, **accdm_params}

PARAM_HASH = hashlib.md5(
    json.dumps(full_params, sort_keys=True, default=str).encode()).hexdigest()[:12]
print('param fingerprint:', PARAM_HASH)

param fingerprint: 7f87a6b0f2f4


## Run CLASS and extract observables

Background is sampled at fixed `z` nodes (interpolated), so the reference is independent of the internal time sampling. `Omega_acc` is the present-day daughter density fraction (last ncdm species).

In [ ]:
# fixed nodes for reproducible sampling
Z_NODES = np.array([0.0, 0.5, 1.0, 2.0, 5.0, 10.0, 30.0, 100.0,
                    300.0, 1000.0, 3e3, 1e4, 1e5, 1e6])
K_NODES = np.logspace(-3, 0.7, 40)   # 1/Mpc

def daughter_key(bg):
    keys = [k for k in bg if k.startswith('(.)rho_ncdm[')]
    return sorted(keys, key=lambda s: int(s.split('[')[1].split(']')[0]))[-1]

def on_z(bg, col, znodes):
    z = np.asarray(bg['z']); order = np.argsort(z)
    return np.interp(znodes, z[order], np.asarray(bg[col])[order])

def run_and_extract(params):
    M = Class(); M.set(params); M.compute()
    bg = M.get_background()
    dkey = daughter_key(bg)
    has_parent = '(.)rho_acc_cdm' in bg
    bgz = {
        'H':            on_z(bg, 'H [1/Mpc]', Z_NODES),
        'rho_acc_cdm':  on_z(bg, '(.)rho_acc_cdm', Z_NODES) if has_parent else np.full_like(Z_NODES, np.nan),
        'rho_daughter': on_z(bg, dkey, Z_NODES),
    }
    rho_crit0 = on_z(bg, '(.)rho_crit', np.array([0.0]))[0]
    omega_acc = float(on_z(bg, dkey, np.array([0.0]))[0] / rho_crit0)

    der = M.get_current_derived_parameters(
        ['sigma8', 'Omega_m', 'h', '100*theta_s', 'z_reio', 'age', 'A_s'])
    der = {k: float(v) for k, v in der.items()}
    der['Omega_acc'] = omega_acc

    cl = M.lensed_cl(LMAX)
    cl_out = {k: np.asarray(cl[k]) for k in ('ell', 'tt', 'te', 'ee', 'pp')}
    pk = np.array([M.pk(float(k), 0.0) for k in K_NODES])

    M.struct_cleanup(); M.empty()
    return {'bgz': bgz, 'scalars': der, 'cl': cl_out, 'pk': pk}

result = run_and_extract(full_params)
print('Omega_acc =', result['scalars']['Omega_acc'])
print('sigma8    =', result['scalars']['sigma8'], '  h =', result['scalars']['h'])

## Reference I/O

In [ ]:
def save_golden(res):
    np.savez(GOLDEN_NPZ,
             z_nodes=Z_NODES, k_nodes=K_NODES,
             ell=res['cl']['ell'],
             cl_tt=res['cl']['tt'], cl_te=res['cl']['te'],
             cl_ee=res['cl']['ee'], cl_pp=res['cl']['pp'],
             pk=res['pk'],
             bg_H=res['bgz']['H'],
             bg_rho_acc_cdm=res['bgz']['rho_acc_cdm'],
             bg_rho_daughter=res['bgz']['rho_daughter'])
    meta = {'param_hash': PARAM_HASH,
            'params':  {k: str(v) for k, v in full_params.items()},
            'scalars': res['scalars'],
            'tol':     TOL}
    GOLDEN_JSON.write_text(json.dumps(meta, indent=2))
    print('wrote golden:', GOLDEN_NPZ, 'and', GOLDEN_JSON)

def load_golden():
    return np.load(GOLDEN_NPZ), json.loads(GOLDEN_JSON.read_text())

generate = (MODE == 'generate') or (MODE == 'auto' and not GOLDEN_NPZ.exists())
if generate:
    if MODE == 'auto':
        print('No golden file found -> generating baseline (review, then commit).')
    save_golden(result)
    ref, meta = load_golden(); is_check = False
else:
    ref, meta = load_golden(); is_check = True
    if meta.get('param_hash') != PARAM_HASH:
        print('WARNING: fiducial params differ from the golden baseline '
              + str(meta.get('param_hash')) + ' != ' + PARAM_HASH + '. '
              + 'A mismatch below is then EXPECTED; regenerate if intentional.')

wrote golden: golden/regression_accDM.npz and golden/regression_accDM.json


## Quantitative checks

Positive quantities (background, P(k), scalars) use element-wise relative deviation; the sign-changing C_l use a max-normalised deviation so zero-crossings (TE) do not blow up. Any FAIL raises an `AssertionError`.

In [ ]:
def dev_elem(cur, refv):
    cur, refv = np.asarray(cur, float), np.asarray(refv, float)
    den = np.where(np.abs(refv) > 0, np.abs(refv), 1.0)
    return np.abs(cur - refv) / den

def dev_maxnorm(cur, refv):
    cur, refv = np.asarray(cur, float), np.asarray(refv, float)
    s = np.max(np.abs(refv)); s = s if s > 0 else 1.0
    return np.abs(cur - refv) / s

report = []
def check(name, cur, refv, tol, kind='elem'):
    d = dev_maxnorm(cur, refv) if kind == 'maxnorm' else dev_elem(cur, refv)
    d = float(np.nanmax(d))
    report.append((name, d, tol, d <= tol))

if is_check:
    for key in ('sigma8', 'Omega_m', 'h', '100*theta_s', 'z_reio', 'age', 'Omega_acc'):
        check('scalar:' + key, result['scalars'][key], meta['scalars'][key], TOL['scalar'])
    check('bg:H',            result['bgz']['H'],            ref['bg_H'],            TOL['bg'])
    check('bg:rho_acc_cdm',  result['bgz']['rho_acc_cdm'],  ref['bg_rho_acc_cdm'],  TOL['bg'])
    check('bg:rho_daughter', result['bgz']['rho_daughter'], ref['bg_rho_daughter'], TOL['bg'])
    msk = result['cl']['ell'] >= 2
    for s in ('tt', 'te', 'ee', 'pp'):
        check('cl:' + s, result['cl'][s][msk], ref['cl_' + s][msk], TOL['cl'], kind='maxnorm')
    check('pk', result['pk'], ref['pk'], TOL['pk'])

    print('{:<20}{:>14}{:>10}   {}'.format('quantity', 'max rel dev', 'tol', 'status'))
    print('-' * 56)
    for name, d, tol, ok in report:
        status = 'PASS' if ok else 'FAIL'
        print('{:<20}{:>14.3e}{:>10.0e}   {}'.format(name, d, tol, status))
    n_fail = sum(1 for *_, ok in report if not ok)
    assert n_fail == 0, str(n_fail) + ' regression check(s) FAILED - see table above.'
    print('\nALL REGRESSION CHECKS PASSED')
else:
    print('Generate mode: baseline written, no comparison performed.')

Generate mode: baseline written, no comparison performed.


## Diagnostic plots

In [ ]:
if is_check:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))

    ax = axes[0, 0]
    msk = result['cl']['ell'] >= 2
    ell = result['cl']['ell'][msk]
    for s, c in zip(('tt', 'te', 'ee', 'pp'), ('C0', 'C1', 'C2', 'C3')):
        cur, rf = result['cl'][s][msk], ref['cl_' + s][msk]
        sc = np.max(np.abs(rf)); sc = sc if sc > 0 else 1.0
        ax.plot(ell, (cur - rf) / sc, c, lw=1, label=s.upper())
    ax.axhline(0, color='k', lw=0.6)
    ax.axhline(TOL['cl'], color='r', ls=':', lw=0.8)
    ax.axhline(-TOL['cl'], color='r', ls=':', lw=0.8)
    ax.set_xlabel('ell'); ax.set_ylabel('rel. dev. vs golden')
    ax.set_title('CMB C_ell'); ax.legend(ncol=2, fontsize=9)
    ax.set_xlim(2, LMAX); ax.set_xscale('log')

    ax = axes[0, 1]
    ax.semilogx(K_NODES, dev_elem(result['pk'], ref['pk']) * np.sign(result['pk'] - ref['pk']),
                'C0', lw=1.2)
    ax.axhline(0, color='k', lw=0.6)
    ax.axhline(TOL['pk'], color='r', ls=':', lw=0.8)
    ax.axhline(-TOL['pk'], color='r', ls=':', lw=0.8)
    ax.set_xlabel('k [1/Mpc]'); ax.set_ylabel('rel. dev.'); ax.set_title('Matter P(k)')

    ax = axes[1, 0]
    ax.loglog(Z_NODES + 1, np.abs(result['bgz']['rho_daughter']), 'C0o-', label='current')
    ax.loglog(Z_NODES + 1, np.abs(ref['bg_rho_daughter']), 'k.--', label='golden')
    ax.set_xlim(1, 1e2)
    ax.set_ylim(1e-14, 1e-6)
    ax.set_xlabel('1+z'); ax.set_ylabel('rho_acc daughter')
    ax.set_title('Daughter density'); ax.legend()

    ax = axes[1, 1]
    names = [r[0] for r in report]; devs = [max(r[1], 1e-18) for r in report]
    cols = ['C2' if r[3] else 'C3' for r in report]
    ax.barh(range(len(names)), devs, color=cols)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
    ax.axvline(TOL['cl'], color='r', ls=':', lw=0.8)
    ax.set_xscale('log'); ax.set_xlabel('max rel dev'); ax.set_title('Per-quantity deviation')

    plt.tight_layout(); plt.show()
else:
    print('Generate mode: re-run with MODE=check (or auto) to see comparison plots.')

Generate mode: re-run with MODE=check (or auto) to see comparison plots.
